In [0]:

data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000),
(103,"Rahul Sharma","Mumbai","Dermatology",1500),
(104,"Priya Nair","Bangalore","Cardiology",5000),
(105,"Vikram Singh","Chennai","Neurology",7000)
]
columns = ["visit_id","patient_name","city","department","consultation_fee"]
df = spark.createDataFrame(data, columns)
display(df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


In [0]:
df.write .mode("overwrite") .parquet("/tmp/patients_parquet")

In [0]:
parquet_df = spark.read.parquet("/tmp/patient_parquet")
display(parquet_df)

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,category
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Medium
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Medium
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,Medium
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Medium
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,High
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Low
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,Low
105,Vikram Singh,Chennai,Neurology,7000,1,7500,High


In [0]:
parquet_df.printSchema()

root
 |-- visit_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- consultation_fee: long (nullable = true)
 |-- tests_count: long (nullable = true)
 |-- total_bill: long (nullable = true)
 |-- category: string (nullable = true)



In [0]:
spark.read.parquet("/tmp/patients_parquet") .select("patient_name","city") .show()

+------------+---------+
|patient_name|     city|
+------------+---------+
| Arjun Reddy|Hyderabad|
|  Priya Nair|Bangalore|
|Rahul Sharma|   Mumbai|
|Vikram Singh|  Chennai|
|Sneha Kapoor|    Delhi|
+------------+---------+



In [0]:
spark.read.parquet("/tmp/patient_parquet") .filter("consultation_fee > 3000") .show()

+--------+------------+---------+----------+----------------+-----------+----------+--------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_bill|category|
+--------+------------+---------+----------+----------------+-----------+----------+--------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          1|      5500|  Medium|
|     107| Karan Patel|Ahmedabad|Cardiology|            5000|          1|      5500|  Medium|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      6000|    High|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|      7500|    High|
+--------+------------+---------+----------+----------------+-----------+----------+--------+



In [0]:
df.write .mode("overwrite") .partitionBy("city") .parquet("/tmp/patients_parquet_partitioned")

In [0]:
spark.read.parquet("/tmp/patients_parquet_partitioned").show()

+--------+------------+-----------+----------------+---------+
|visit_id|patient_name| department|consultation_fee|     city|
+--------+------------+-----------+----------------+---------+
|     103|Rahul Sharma|Dermatology|            1500|   Mumbai|
|     102|Sneha Kapoor|Orthopedics|            3000|    Delhi|
|     101| Arjun Reddy| Cardiology|            5000|Hyderabad|
|     105|Vikram Singh|  Neurology|            7000|  Chennai|
|     104|  Priya Nair| Cardiology|            5000|Bangalore|
+--------+------------+-----------+----------------+---------+



In [0]:
spark.read.parquet("/tmp/patients_parquet_partitioned") .filter("city = 'Hyderabad'") .show()

+--------+------------+----------+----------------+---------+
|visit_id|patient_name|department|consultation_fee|     city|
+--------+------------+----------+----------------+---------+
|     101| Arjun Reddy|Cardiology|            5000|Hyderabad|
+--------+------------+----------+----------------+---------+



In [0]:
new_data = [(106,"Ananya Das","Kolkata","Orthopedics",3000)]
new_df = spark.createDataFrame(new_data, columns)
new_df.write .mode("append") .parquet("/tmp/patients_parquet")

In [0]:
df.write .mode("overwrite") .parquet("/tmp/patients_parquet")

In [0]:
%sql
CREATE OR REPLACE TABLE patients_parquet_table
USING DELTA
AS
SELECT * FROM parquet.`/tmp/patients_parquet`;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM patients_parquet_table;

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


In [0]:
spark.sql("""
CONVERT TO DELTA parquet.`/tmp/patients_parquet`
""")

DataFrame[]

In [0]:
%sql
UPDATE patients_parquet_table SET consultation_fee = 6000 WHERE visit_id = 101;

num_affected_rows
1


In [0]:
%sql
UPDATE delta.`/tmp/patient_parquet` SET consultation_fee = 6000 WHERE visit_id = 101;

num_affected_rows
1


In [0]:
df.write.mode("overwrite").parquet("dbfs:/FileStore/patient_parquet")

In [0]:
parquet_df = spark.read.parquet("dbfs:/FileStore/patient_parquet")
parquet_df.show()

+--------+------------+---------+-----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|
+--------+------------+---------+-----------+----------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|
+--------+------------+---------+-----------+----------------+



In [0]:
parquet_df.filter("consultation_fee > 3000").show()

+--------+------------+---------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|
+--------+------------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|
+--------+------------+---------+----------+----------------+



In [0]:
df.write .mode("overwrite") .partitionBy("city") .parquet("dbfs:/FileStore/patient_parquet_partitioned")

In [0]:
spark.read.parquet("dbfs:/FileStore/patient_parquet_partitioned") .filter("city = 'Hyderabad'") .show()

+--------+------------+----------+----------------+---------+
|visit_id|patient_name|department|consultation_fee|     city|
+--------+------------+----------+----------------+---------+
|     101| Arjun Reddy|Cardiology|            5000|Hyderabad|
+--------+------------+----------+----------------+---------+



In [0]:
new_df.write.mode("append").parquet("dbfs:/FileStore/patient_parquet")

In [0]:
df.write .format("delta") .mode("overwrite") .saveAsTable("patients_parquet_table")

In [0]:
%sql
UPDATE patients_parquet_table SET consultation_fee = 6000 WHERE visit_id = 101;

num_affected_rows
1


In [0]:
Difference between Parquet and Delta?

Parquet is a columnar storage format optimized for fast read performance and efficient compression. However, it does not support updates, deletes, or transactions—it is mainly used for storing and querying large datasets.

Delta Lake, on the other hand, is built on top of Parquet and adds a transaction layer. It provides ACID transactions, schema enforcement, and supports operations like update, delete, and merge. This makes Delta more reliable and suitable for production data pipelines.